# Week 85, Day 1: Review: testing

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w085d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def average(xs):
    return sum(xs) / len(xs)

def test_average():
    assert average([1, 2, 3]), 2      # spot the problem

try:
    test_average()
    print("passed")
except AssertionError:
    print("failed")
print(average([1, 2, 3]) == 2, 0.1 + 0.2 == 0.3)

In [ ]:
import math

def split(total, people):
    if people < 1:
        raise ValueError("at least one person")
    share, extra = divmod(total, people)
    return [share + 1] * extra + [share] * (people - extra)

cases = [(100, 3, [34, 33, 33]), (99, 3, [33, 33, 33]), (5, 1, [5])]
for total, people, expected in cases:
    result = split(total, people)
    assert result == expected and sum(result) == total, (total, people, result)
try:
    split(10, 0)
except ValueError as e:
    print("raised:", e)
print(math.isclose(0.1 + 0.2, 0.3), "all cases passed")

In [ ]:
import random
from unittest.mock import Mock

def notify_all(users, send):
    for u in users:
        if u["opted_in"]:
            send(u["email"], "Your report is ready")

send = Mock()
notify_all([{"email": "a@example.com", "opted_in": True}, {"email": "b@example.com", "opted_in": False}], send)
send.assert_called_once_with("a@example.com", "Your report is ready")
print(send.call_count, send.call_args_list)

def dedupe(xs):
    return list(dict.fromkeys(xs))

rng = random.Random(0)
for _ in range(200):
    xs = [rng.randint(0, 5) for _ in range(rng.randint(0, 8))]
    out = dedupe(xs)
    assert set(out) == set(xs) and len(out) == len(set(xs)) and out == sorted(out, key=xs.index)
print("properties held for 200 random lists")

In [ ]:
import tomllib

text = '''
[project]
name = "spend"
version = "1.4.0"
requires-python = ">=3.11"
dependencies = ["requests>=2.31", "rich"]
'''
project = tomllib.loads(text)["project"]
print(project["name"], project["version"], project["dependencies"])

def parse(v):
    return tuple(int(part) for part in v.split("."))

print("1.10.0" < "1.9.0", parse("1.10.0") < parse("1.9.0"))
print(sorted(["1.10.0", "1.9.0", "1.9.12", "2.0.0"], key=parse))

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript('''
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, total REAL);
INSERT INTO customers VALUES (1, 'Ada'), (2, 'Bo'), (3, 'Cy');
INSERT INTO orders VALUES (1, 1, 20.0), (2, 1, 15.5), (3, 2, 8.0);
''')
rows = conn.execute('''
    SELECT c.name, COUNT(o.id), COALESCE(SUM(o.total), 0)
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.id
    GROUP BY c.id ORDER BY 3 DESC
''').fetchall()
print(rows)
name = "O'Brien"
conn.execute("INSERT INTO customers (name) VALUES (?)", (name,))
print(conn.execute("SELECT id FROM customers WHERE name = ?", (name,)).fetchone())
try:
    with conn:
        conn.execute("UPDATE orders SET total = total - 100 WHERE id = 1")
        raise ValueError("payment failed")
except ValueError:
    pass
print(conn.execute("SELECT total FROM orders WHERE id = 1").fetchone())

> **Key idea.** Good tests catch bugs: parametrize cases with ids, set up with fixtures (clean-up after `yield`), check errors with `pytest.raises(match=...)` and floats with `approx`; coverage shows what ran, mutation testing shows what's checked. Replace awkward dependencies with mocks, fakes or, simplest, injected parameters, and test properties that hold for every input. `pyproject.toml` declares a project and its requirements; semantic versions compare as number tuples, `^` allows compatible minor updates and `~` patches only; libraries use ranges, applications pin. Query with joins, `GROUP BY` and `HAVING`, always through `?` parameters, and wrap related changes in `with conn:` so they commit or roll back together.

### Exercise: Tests for splitting a bill

`bill.py` has `split(total, people, tip=0)`, which adds a tip (a percentage, rounded to the nearest penny) to a total in pence and splits it between `people` so that shares differ by at most a penny, larger shares first. It raises `ValueError` for fewer than one person or a negative amount. Write tests that pass on this code and catch every broken version the checks try (six of them).

In [ ]:
# Run this cell first: it creates the files used below.
with open('bill.py', 'w', encoding='utf-8') as f:
    f.write('def split(total, people, tip=0):\n    if people < 1:\n        raise ValueError("at least one person")\n    if total < 0 or tip < 0:\n        raise ValueError("no negative amounts")\n    grand = total + round(total * tip / 100)\n    share, extra = divmod(grand, people)\n    return [share + 1] * extra + [share] * (people - extra)\n')

In [ ]:
import pytest
from bill import split

def test_even_split():
    assert split(900, 3) == [300, 300, 300]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"bill.py\").read()\nBROKEN = {\n    \"ignores the tip\": GOOD.replace(\"grand = total + round(total * tip / 100)\", \"grand = total\"),\n    \"truncates the tip\": GOOD.replace(\"round(total * tip / 100)\", \"int(total * tip / 100)\"),\n    \"allows nobody\": GOOD.replace(\"if people < 1:\", \"if people < 0:\"),\n    \"allows negatives\": GOOD.replace(\"if total < 0 or tip < 0:\", \"if False:\"),\n    \"extra pennies last\": GOOD.replace(\"[share + 1] * extra + [share] * (people - extra)\", \"[share] * (people - extra) + [share + 1] * extra\"),\n    \"loses the remainder\": GOOD.replace(\"[share + 1] * extra + [share] * (people - extra)\", \"[share] * people\"),\n}\ndef test_catches_everything():\n    \"every broken version caught\"\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"bill\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"bill.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: Tests with a mock

`reminders.py` has `send_due(reminders, now, send)`: for each reminder (a dict with `email`, `text` and `due`) that is due (`due <= now`) and not yet `sent`, it calls `send(email, text)`, marks it `sent`, and returns how many it sent. Write tests that pass `unittest.mock.Mock()` as `send` and catch every broken version (five).

In [ ]:
# Run this cell first: it creates the files used below.
with open('reminders.py', 'w', encoding='utf-8') as f:
    f.write('def send_due(reminders, now, send):\n    count = 0\n    for r in reminders:\n        if r["due"] <= now and not r.get("sent"):\n            send(r["email"], r["text"])\n            r["sent"] = True\n            count += 1\n    return count\n')

In [ ]:
from unittest.mock import Mock
from reminders import send_due

def test_sends_something():
    send = Mock()
    send_due([{"email": "a@example.com", "text": "hi", "due": 1}], 5, send)
    assert send.called

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"reminders.py\").read()\nBROKEN = {\n    \"misses the due moment\": GOOD.replace('r[\"due\"] <= now', 'r[\"due\"] < now'),\n    \"sends again\": GOOD.replace(' and not r.get(\"sent\")', \"\"),\n    \"forgets to mark\": GOOD.replace('            r[\"sent\"] = True\\n', \"\"),\n    \"swaps the arguments\": GOOD.replace('send(r[\"email\"], r[\"text\"])', 'send(r[\"text\"], r[\"email\"])'),\n    \"counts everything\": GOOD.replace(\"return count\", \"return len(reminders)\"),\n}\ndef test_mocks():\n    \"every broken version caught\"\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"reminders\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"reminders.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: Properties and shrinking

Write `counterexample(fn, make_input, prop, trials, rng)`: call `make_input(rng)` up to `trials` times and return the first input for which `prop(x, fn(x))` is false (or `None`). Then `shrink(xs, still_fails)`: given a failing list, repeatedly try removing one element at a time (from the front) and keep any smaller list for which `still_fails` is true, until no single removal still fails; return the smallest list found.

In [ ]:
import random

def counterexample(fn, make_input, prop, trials, rng):
    ___

def shrink(xs, still_fails):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\n\ndef bad_sort(xs):\n    return sorted(set(xs))          # loses duplicates\n\ndef test_properties():\n    \"find a failure, then make it small\"\n    make = lambda rng: [rng.randint(0, 9) for _ in range(rng.randint(0, 10))]\n    prop = lambda xs, out: out == sorted(xs)\n    x = counterexample(bad_sort, make, prop, 500, random.Random(1))\n    assert x is not None and len(x) != len(set(x))\n    assert counterexample(sorted, make, prop, 300, random.Random(1)) is None\n    small = shrink(x, lambda xs: bad_sort(xs) != sorted(xs))\n    assert len(small) == 2 and small[0] == small[1], small\n    assert shrink([5, 1, 7], lambda xs: 7 in xs) == [7]")

### Exercise: Semantic versions

Write `parse(v)`: `"1.4.2"` to `(1, 4, 2)`, raising `ValueError("bad version X")` for anything that isn't three whole numbers. `bump(v, part)` for `"major"`, `"minor"` or `"patch"` (resetting the parts to its right). `satisfies(v, spec)`: `spec` is one or more clauses separated by commas, all of which must hold: `^X.Y.Z` (at least that version, and below the next major, or for `0.Y.Z` below the next minor), `~X.Y.Z` (at least that, below the next minor), and the operators `>=`, `>`, `<=`, `<`, `==` and `!=`. And `best(versions, spec)`: the highest version satisfying `spec`, or `None`.

In [ ]:
def parse(v):
    ___

def bump(v, part):
    ___

def satisfies(v, spec):
    ___

def best(versions, spec):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parse_bump():\n    \"numbers, not text\"\n    assert parse(\"1.10.0\") > parse(\"1.9.0\") and parse(\" 2.0.1 \") == (2, 0, 1)\n    for bad in (\"1.2\", \"1.2.x\", \"1.2.3.4\"):\n        try:\n            parse(bad)\n        except ValueError as e:\n            assert str(e) == f\"bad version {bad}\"\n        else:\n            raise AssertionError(bad)\n    assert bump(\"1.4.2\", \"patch\") == \"1.4.3\" and bump(\"1.4.2\", \"minor\") == \"1.5.0\" and bump(\"1.4.2\", \"major\") == \"2.0.0\"\n\ndef test_ranges():\n    \"caret, tilde and operators\"\n    assert satisfies(\"1.9.3\", \"^1.2.0\") and not satisfies(\"2.0.0\", \"^1.2.0\") and not satisfies(\"1.1.9\", \"^1.2.0\")\n    assert satisfies(\"0.3.9\", \"^0.3.1\") and not satisfies(\"0.4.0\", \"^0.3.1\")\n    assert satisfies(\"1.2.9\", \"~1.2.0\") and not satisfies(\"1.3.0\", \"~1.2.0\")\n    assert satisfies(\"1.5.0\", \">=1.0.0, <2.0.0, !=1.5.1\") and not satisfies(\"1.5.1\", \">=1.0.0,<2.0.0,!=1.5.1\")\n    assert satisfies(\"3.0.0\", \"==3.0.0\") and satisfies(\"3.0.1\", \">3.0.0\") and satisfies(\"3.0.0\", \"<=3.0.0\")\n    assert best([\"1.2.0\", \"1.10.1\", \"1.9.9\", \"2.0.0\"], \"^1.2.0\") == \"1.10.1\" and best([\"0.9.0\"], \">=1.0.0\") is None")

### Exercise: Queries and a transaction

The given `make_db()` builds an in-memory shop database. Write `top_customers(conn, n)`: the `n` biggest spenders as `(name, orders, spent)` with `spent` rounded to 2 places, biggest first (ties by name), using one query with a join, `GROUP BY` and a `LIMIT ?` parameter. `busy_cities(conn, min_orders)`: the sorted cities whose customers placed at least `min_orders` orders (use `HAVING`). And `transfer_stock(conn, product, from_shop, to_shop, qty)`: move stock in **one transaction**: if `from_shop` has fewer than `qty`, raise `ValueError("not enough stock")` and change nothing; create the destination row if it doesn't exist.

In [ ]:
import sqlite3

def make_db():
    conn = sqlite3.connect(":memory:")
    conn.executescript('''
    CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT);
    CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER REFERENCES customers(id), total REAL);
    CREATE TABLE stock (shop TEXT, product TEXT, qty INTEGER, PRIMARY KEY (shop, product));
    INSERT INTO customers VALUES (1, 'Ada', 'York'), (2, 'Bo', 'Leeds'), (3, 'Cy', 'York'), (4, 'Dee', 'Hull');
    INSERT INTO orders VALUES (1, 1, 20.0), (2, 1, 15.5), (3, 2, 40.0), (4, 3, 5.25), (5, 3, 10.0), (6, 2, 0.5);
    INSERT INTO stock VALUES ('York', 'mug', 10), ('Leeds', 'mug', 2), ('York', 'lamp', 1);
    ''')
    return conn

def top_customers(conn, n):
    ___

def busy_cities(conn, min_orders):
    ___

def transfer_stock(conn, product, from_shop, to_shop, qty):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _stock(conn):\n    return conn.execute(\"SELECT shop, product, qty FROM stock ORDER BY shop, product\").fetchall()\n\ndef test_queries():\n    \"joins, groups and parameters\"\n    conn = make_db()\n    assert top_customers(conn, 2) == [(\"Bo\", 2, 40.5), (\"Ada\", 2, 35.5)]\n    assert top_customers(conn, 10)[-1] == (\"Cy\", 2, 15.25) and len(top_customers(conn, 10)) == 3\n    assert busy_cities(conn, 4) == [\"York\"] and busy_cities(conn, 2) == [\"Leeds\", \"York\"] and busy_cities(conn, 9) == []\n    assert \"?\" in source and \"LIMIT\" in source.upper() and \"HAVING\" in source.upper()\n\ndef test_transfer():\n    \"all or nothing\"\n    conn = make_db()\n    transfer_stock(conn, \"mug\", \"York\", \"Leeds\", 3)\n    transfer_stock(conn, \"lamp\", \"York\", \"Hull\", 1)\n    assert _stock(conn) == [(\"Hull\", \"lamp\", 1), (\"Leeds\", \"mug\", 5), (\"York\", \"lamp\", 0), (\"York\", \"mug\", 7)]\n    before = _stock(conn)\n    try:\n        transfer_stock(conn, \"mug\", \"Leeds\", \"York\", 6)\n    except ValueError as e:\n        assert str(e) == \"not enough stock\"\n    else:\n        raise AssertionError(\"should refuse\")\n    assert _stock(conn) == before\n    try:\n        transfer_stock(conn, \"kettle\", \"York\", \"Leeds\", 1)\n    except ValueError:\n        pass\n    assert _stock(conn) == before")

### Exercise: Reading pyproject.toml

Write `summary(path)`: read the `pyproject.toml` with `tomllib` (open it in binary mode) and return a dict with `name`, `version`, `python` (the `requires-python` value), `dependencies` (the **names** of the required packages, sorted, without version specifiers or markers) and `scripts` (the `[project.scripts]` table). Then `unpinned(path)`: the sorted names of every dependency, including optional ones, that has **no** version specifier at all.

In [ ]:
# Run this cell first: it creates the files used below.
with open('pyproject.toml', 'w', encoding='utf-8') as f:
    f.write('[project]\nname = "spend"\nversion = "1.4.0"\nrequires-python = ">=3.11"\ndependencies = [\n    "requests>=2.31",\n    "rich",\n    "click==8.1.7",\n    "tomli; python_version < \'3.11\'",\n]\n\n[project.optional-dependencies]\ndev = ["pytest>=8", "ruff"]\n\n[project.scripts]\nspend = "spend.cli:main"\n')

In [ ]:
import re
import tomllib

def summary(path):
    ___

def unpinned(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pyproject():\n    \"names, versions and entry points\"\n    s = summary(\"pyproject.toml\")\n    assert s == {\"name\": \"spend\", \"version\": \"1.4.0\", \"python\": \">=3.11\",\n                 \"dependencies\": [\"click\", \"requests\", \"rich\", \"tomli\"], \"scripts\": {\"spend\": \"spend.cli:main\"}}\n    assert unpinned(\"pyproject.toml\") == [\"rich\", \"ruff\", \"tomli\"]")

### Bug hunt: A test that can't fail

The test asserts that `total(...)` is **truthy**, with `30` as the message, so it passes even though `total` is wrong: this prints `caught: False`. Make the test compare the result with 30, so it catches the bug and prints `caught: True`.

In [ ]:
def total(prices):
    return sum(prices[1:])          # the bug: skips the first price

def test_total():
    assert total([10, 20]), 30

try:
    test_total()
    print("caught: False")
except AssertionError:
    print("caught: True")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"compare, don't just assert truthiness\"\n    expect_output(\"caught: True\")")

### Bug hunt: A correct answer that fails

The code is right, but the test compares floats exactly, so it fails: `failed`. Compare with a tolerance (`math.isclose`) so it prints `passed`.

In [ ]:
import math

def vat(price):
    return price * 1.2

try:
    assert vat(3.3) == 3.96
    print("passed")
except AssertionError:
    print("failed")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"isclose\"\n    expect_output(\"passed\")")

### Bug hunt: An apostrophe breaks the query

The query is built with an f-string, so the apostrophe in "O'Brien" ends the SQL string early and it crashes with `sqlite3.OperationalError`. Use a `?` parameter so it prints `found: [(1,)]`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE people (id INTEGER PRIMARY KEY, name TEXT)")
conn.execute("INSERT INTO people (name) VALUES (?)", ("O'Brien",))
name = "O'Brien"
print("found:", conn.execute(f"SELECT id FROM people WHERE name = '{name}'").fetchall())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"parameters, never formatting\"\n    expect_output(\"found: [(1,)]\")")

### Puzzle: A fixture with clean-up

Arrange a pytest fixture that creates a database for each test and always closes it, and a test that uses it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: "1.10.0" < "1.9.0" as Python strings
q = ___   # the next version after "2.3.7" for a backwards-compatible new feature
r = ___   # True or False: "^1.4.0" allows "1.12.3"
s = ___   # the SQL keyword that filters groups after GROUP BY
t = ___   # the number of rows "SELECT * FROM a LEFT JOIN b ON ..." returns for 3 rows in a, none matching

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q == \"2.4.0\" and r is True, \"p, q and r\"\n    assert s.upper() == \"HAVING\" and t == 3, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a suite that catches every mutant

`inventory.py` has a `Stock` class: `add(sku, qty)` (positive quantities only), `available(sku)` (stock minus reservations), `reserve(sku, qty)` (positive, and no more than is available), `release(sku, qty)` (undo a reservation; no more than is reserved) and `ship(sku, qty)` (send reserved stock: it leaves both the reservation and the stock). Errors are `ValueError`s. Write a test suite, using a fixture for a stocked `Stock`, that passes on this code and catches **all eight** broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('inventory.py', 'w', encoding='utf-8') as f:
    f.write('class Stock:\n    def __init__(self):\n        self._items = {}\n        self._reserved = {}\n\n    def add(self, sku, qty):\n        if qty <= 0:\n            raise ValueError("quantity must be positive")\n        self._items[sku] = self._items.get(sku, 0) + qty\n\n    def available(self, sku):\n        return self._items.get(sku, 0) - self._reserved.get(sku, 0)\n\n    def reserve(self, sku, qty):\n        if qty <= 0:\n            raise ValueError("quantity must be positive")\n        if qty > self.available(sku):\n            raise ValueError("not enough stock")\n        self._reserved[sku] = self._reserved.get(sku, 0) + qty\n\n    def release(self, sku, qty):\n        if qty > self._reserved.get(sku, 0):\n            raise ValueError("not reserved")\n        self._reserved[sku] -= qty\n\n    def ship(self, sku, qty):\n        if qty > self._reserved.get(sku, 0):\n            raise ValueError("not reserved")\n        self._reserved[sku] -= qty\n        self._items[sku] -= qty\n')

In [ ]:
import pytest
from inventory import Stock

@pytest.fixture
def stock():
    s = Stock()
    s.add("mug", 10)
    return s

def test_add(stock):
    assert stock.available("mug") == 10

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"inventory.py\").read()\nBROKEN = {\n    \"add allows zero\": GOOD.replace(\"        if qty <= 0:\\n            raise ValueError(\\\"quantity must be positive\\\")\\n        self._items\", \"        if qty < 0:\\n            raise ValueError(\\\"quantity must be positive\\\")\\n        self._items\"),\n    \"add overwrites\": GOOD.replace(\"self._items[sku] = self._items.get(sku, 0) + qty\", \"self._items[sku] = qty\"),\n    \"available ignores reservations\": GOOD.replace(\"return self._items.get(sku, 0) - self._reserved.get(sku, 0)\", \"return self._items.get(sku, 0)\"),\n    \"can't reserve the last one\": GOOD.replace(\"if qty > self.available(sku):\", \"if qty >= self.available(sku):\"),\n    \"reserves one too many\": GOOD.replace(\"if qty > self.available(sku):\", \"if qty > self.available(sku) + 1:\"),\n    \"release does nothing\": GOOD.replace(\"        self._reserved[sku] -= qty\\n\\n    def ship\", \"        pass\\n\\n    def ship\"),\n    \"ship keeps the stock\": GOOD.replace(\"        self._items[sku] -= qty\", \"        pass\"),\n    \"ship keeps the reservation\": GOOD.replace(\"        self._reserved[sku] -= qty\\n        self._items[sku] -= qty\", \"        self._items[sku] -= qty\"),\n}\ndef test_suite():\n    \"all eight caught\"\n    for name, code in BROKEN.items():\n        assert code != GOOD, f\"mutant {name} didn't apply\"\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"inventory\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"inventory.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"\n    assert \"@pytest.fixture\" in source")

## How to review

For each section: read the summary, predict the example's output **before** running it, and only then check. Then do the exercises without looking back at the weeks they come from; when you get stuck, note the topic, look it up, and come back. The notes you make are your revision list for Friday.

The test passes, but it would pass for almost any `average`: `assert x, 2` asserts that `x` is truthy and uses `2` as the failure message. A test is only worth what it would catch. If that surprised you, start with Week 72.

## Tests that catch bugs

- **pytest** (Week 72) runs functions named `test_*` and reports plain `assert` failures with the values involved.
- **`@pytest.mark.parametrize("a, b", [...], ids=[...])`** runs one test over many cases, each named in the report.
- **Fixtures** set up what tests need: a function decorated with `@pytest.fixture`, requested by naming it as a parameter. Code after a `yield` in a fixture is the clean-up. Built-ins include `tmp_path` (a fresh folder) and `monkeypatch`.
- **`pytest.raises(ValueError, match="...")`** checks that an error happens (and its message); **`pytest.approx`** compares floats with a tolerance.
- **Coverage** (Week 72, Day 4) tells you which lines ran, not whether anything checked them. **Mutation testing** asks the better question: if the code were subtly wrong (a `<` for a `<=`, a dropped line), would some test fail? Every "write tests that catch every broken version" exercise in this course is mutation testing.

## Doubles and properties

- A **test double** stands in for something awkward: a **mock** (`unittest.mock.Mock`) records how it was called (`assert_called_once_with`, `call_args_list`, `call_count`); a **fake** is a small working substitute (an in-memory store); a **stub** returns canned answers. `patch("module.name")` swaps one in temporarily; often simpler is to **inject** the dependency (pass `send`, `clock` or `rng` as a parameter), which needs no patching at all.
- **Property-based testing** (Week 72, Day 3) checks rules that hold for **every** input instead of a few chosen ones: a sorted list has the same items and is in order; decoding an encoding gives back the original; a fast function agrees with a slow, obviously correct one. Hypothesis generates the inputs and **shrinks** a failure to the smallest example; the idea works with any random generator.

## Packages and versions

- A project's **`pyproject.toml`** (Week 73) names it and lists what it needs: `[project]` with `name`, `version`, `requires-python`, `dependencies` (each a **requirement** like `"requests>=2.31"`), `[project.optional-dependencies]` (extras such as `dev`), and `[project.scripts]` for command-line entry points. `tomllib` (standard library) reads it.
- **Semantic versioning**: `MAJOR.MINOR.PATCH`. Bump PATCH for fixes, MINOR for new features that don't break anything, MAJOR for breaking changes (and reset the numbers to its right). Compare versions as tuples of numbers, never as text (`"1.10.0" < "1.9.0"` as strings).
- **Ranges**: `^1.2.0` means "compatible with 1.2.0" (at least 1.2.0, below 2.0.0); `~1.2.0` allows patches only (below 1.3.0). Libraries declare ranges; applications **pin** exact versions in a lock file so every install is the same.
- **Code quality** (Week 74): PEP 8 names (`snake_case` functions, `CapWords` classes), docstrings, small functions, review, and CI that runs the linter and tests on every push.

## Databases

- **SQL** (Week 75): `SELECT columns FROM table WHERE condition ORDER BY ... LIMIT n`; `JOIN ... ON` combines tables; `GROUP BY` with `COUNT`, `SUM`, `AVG` summarises; `HAVING` filters groups (after grouping), `WHERE` filters rows (before).
- **`sqlite3`**: `conn.execute(sql, params)` with **`?` placeholders**, never string formatting: parameters can't change the query's meaning (Week 79's injection attacks) and handle quotes for you. `fetchall()`, `fetchone()`.
- **Transactions**: `with conn:` commits if the block finishes and rolls back if it raises, so a multi-step change (move stock from one shop to another) happens completely or not at all.
- **Indexes** make lookups on a column fast at the cost of slower writes; **ORMs** (SQLAlchemy) map rows to objects but generate the same SQL underneath.

The `LEFT JOIN` keeps Cy, who has no orders; the quote in "O'Brien" needs no escaping with a parameter; and the failed transaction left the order untouched.

## Classic mistakes

- **Tests that can't fail**: `assert x, expected`, a test that never calls the code, or `pytest.raises` around nothing. Make a test fail once, on purpose, to see it works.
- **Testing the mock**: a test that asserts only what the mock was set to return checks nothing about your code.
- **Exact float comparisons** in tests: use `pytest.approx` or `math.isclose`.
- **Comparing versions as strings**, or pinning a library's dependencies exactly (which makes it impossible to install alongside others).
- **Building SQL with f-strings**, forgetting `with conn:` around related changes, and filtering groups with `WHERE` instead of `HAVING`.

## Practice

## Quiz

## Daily challenge

## Before you go

Write down the three topics from today you'd least like to see in Friday's exam, and for each, the week to revisit. Then redo one exercise from each of those weeks without looking at the solution. Tomorrow reviews **the web and security**: HTTP, APIs and frameworks, automation, and secure coding.